# 00 · Descarga e inventario de Polyvore

Descarga el dataset de Han et al. (2017), repositorio [`xthan/polyvore-dataset`](https://github.com/xthan/polyvore-dataset) (licencia Apache 2.0). Solo metadatos en JSON, sin imágenes.

**Salidas:** `data/external/polyvore/` y `data/processed/hito3/polyvore_items.csv`, `polyvore_outfits.csv` (no se suben a git; este notebook los regenera).

**Entrada editable:** `data/mappings/polyvore_categorias.csv`, la tabla categoría → slot.

> Cita: Han, X., Wu, Z., Jiang, Y.-G. y Davis, L. S. (2017). *Learning Fashion Compatibility with Bidirectional LSTMs*. ACM Multimedia.

In [1]:
import hashlib
import json
import re
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")

DESTINO = ROOT / "data" / "external" / "polyvore"
DESTINO.mkdir(parents=True, exist_ok=True)
BASE = "https://github.com/xthan/polyvore-dataset/raw/master/"
ARCHIVOS = ["polyvore.tar.gz", "category_id.txt", "LICENSE"]
SHA256_TAR = "37c36efc67cabdaa58f2d73cbc1c5143d87f9918ff3d4da20e271e40c89e902b"
CONTENIDO = ["train_no_dup.json", "valid_no_dup.json", "test_no_dup.json",
             "fill_in_blank_test.json", "fashion_compatibility_prediction.txt"]

## 1. Descarga y verificación

In [2]:
for nombre in ARCHIVOS:
    ruta = DESTINO / nombre
    if not ruta.exists():
        urllib.request.urlretrieve(BASE + nombre, ruta)
        print("Descargado:", nombre)

sha = hashlib.sha256((DESTINO / "polyvore.tar.gz").read_bytes()).hexdigest()
if sha != SHA256_TAR:
    raise ValueError(f"polyvore.tar.gz no coincide con el archivo verificado (sha256 {sha}).")

with tarfile.open(DESTINO / "polyvore.tar.gz") as tar:
    nombres = tar.getnames()
    if sorted(nombres) != sorted(CONTENIDO):
        raise ValueError(f"Contenido inesperado en el tar: {nombres}")
    for nombre in CONTENIDO:
        if not (DESTINO / nombre).exists():
            tar.extract(nombre, DESTINO, filter="data")
print("OK:", sorted(p.name for p in DESTINO.iterdir()))

OK: ['LICENSE', 'category_id.txt', 'fashion_compatibility_prediction.txt', 'fill_in_blank_test.json', 'polyvore.tar.gz', 'test_no_dup.json', 'train_no_dup.json', 'valid_no_dup.json']


## 2. Inventario

In [3]:
def tid(item):
    """Id del producto en Polyvore, sacado de la URL de su imagen."""
    return re.search(r"tid=(\d+)", item["image"]).group(1)

particiones = {s: json.loads((DESTINO / f"{s}_no_dup.json").read_text(encoding="utf-8"))
               for s in ["train", "valid", "test"]}
categorias = dict(
    linea.split(" ", 1) for linea in (DESTINO / "category_id.txt").read_text(encoding="utf-8").splitlines() if linea.strip()
)

filas, productos = [], {}
for s, outfits in particiones.items():
    items = [it for o in outfits for it in o["items"]]
    for it in items:
        productos.setdefault(tid(it), set()).add(s)
    filas.append({
        "particion": s,
        "outfits": len(outfits),
        "items": len(items),
        "productos_unicos": len({tid(it) for it in items}),
        "items_sin_nombre": sum(not it["name"].strip() for it in items),
    })
inventario = pd.DataFrame(filas)
display(inventario)
print("Productos que aparecen en más de una partición oficial:",
      sum(len(v) > 1 for v in productos.values()), "de", len(productos))

,particion,outfits,items,productos_unicos,items_sin_nombre
0,train,17316,114806,84497,1317
1,valid,1497,9070,8923,202
2,test,3076,18604,18169,352


Productos que aparecen en más de una partición oficial: 0 de 111589


In [4]:
largo = Counter(len(o["items"]) for outfits in particiones.values() for o in outfits)
display(pd.Series(largo, name="outfits").sort_index().rename_axis("items por outfit").to_frame())

cat_uso = Counter(str(it["categoryid"]) for outfits in particiones.values() for o in outfits for it in o["items"])
top = pd.DataFrame(cat_uso.most_common(25), columns=["categoryid", "items"])
top["nombre"] = top["categoryid"].map(categorias)
print(f"Categorías usadas: {len(cat_uso)} (de {len(categorias)} en category_id.txt). Las 25 más frecuentes:")
display(top)

sin_nombre = sorted(set(cat_uso) - set(categorias))
print("categoryid sin nombre en category_id.txt:", sin_nombre)

,outfits
items por outfit,
4,2400
5,3691
6,4028
7,3903
8,7867


Categorías usadas: 380 (de 380 en category_id.txt). Las 25 más frecuentes:


,categoryid,items,nombre
0,64,5118,Earrings
1,37,5098,Shoulder Bags
2,62,4719,Necklaces
3,11,4226,Tops
4,43,4113,Pumps
5,46,4101,Sandals
6,57,4042,Sunglasses
7,261,3904,Ankle Booties
8,106,3540,Bracelets & Bangles
9,38,3459,Clutches


categoryid sin nombre en category_id.txt: []


In [5]:
ejemplo = particiones["train"][0]
print(ejemplo["name"], "|", ejemplo["desc"])
display(pd.DataFrame(ejemplo["items"])[["index", "name", "categoryid", "price", "likes"]]
        .assign(categoria=lambda t: t["categoryid"].astype(str).map(categorias)))

Casual | A fashion look from January 2017 by beebeely-look featuring Fuji, Citizens of Humanity, casual, casualoutfit, Packandgo, winterjacket and gearbest


,index,name,categoryid,price,likes,categoria
0,1,mock neck embroidery suede sweatshirt,4495,24.0,10,Sweatshirts
1,2,luxe double zip hooded jacket,25,150.0,2250,Jackets
2,3,citizens humanity high rise rocket hem jean,27,248.0,2437,Jeans
3,4,suede tie short boots,261,37.0,2,Ankle Booties
4,5,cloth travel school backpack,259,22.0,2,Backpacks
5,6,,1967,160.0,3,Tech Accessories
6,7,polyvore,2,-1.0,51,Clothing


## 3. Paso 1 · Categorías → slot

Cada una de las 380 categorías de Polyvore se asigna a un slot del esquema común (`arriba`, `abajo`, `vestido`, `exterior`, `calzado`, `bolso`, `accesorio`, `joyeria`) o se marca `excluido` con su motivo. La tabla está en `data/mappings/polyvore_categorias.csv` y **se revisa y corrige a mano**; este notebook solo la aplica.

Decisiones de la tabla (declararlas en el informe):
- **Cárdigan, chaleco y blazer** → `exterior` (son capas sobre la prenda de arriba).
- **Enterizo y jumpsuit** → `vestido` (pieza única que reemplaza arriba + abajo).
- **Medias, panties y calcetines** → `accesorio`.
- **Relojes** → `accesorio`; **joyería** aparte.
- **Billeteras** (*Wallets*, *Men's Bags & Wallets*) → `accesorio` (revisión del equipo, 01/11/2026).
- **Trajes** (*Suits*) → `arriba` y **ropa de baño de hombre** (*Men's Swimwear*) → `abajo` (revisión del equipo, 01/11/2026).
- Se excluyen: belleza, hogar, tecnología, comida y papelería; categorías genéricas o ambiguas (*Clothing*, *Men's Clothing*, *Activewear*, *Costumes*); ropa interior y de dormir; ropa de baño de mujer; niños y maternidad.
- El **largo** se toma de la categoría cuando la fija (*Mini Skirts* → corto, *Knee Length Skirts* → midi, *Long Skirts* y *Gowns* → largo, *Shorts* → corto, *Capri* → midi). El resto se saca del texto en el paso 3.

In [6]:
sys_path = str(ROOT / "src")
import sys
if sys_path not in sys.path:
    sys.path.insert(0, sys_path)
from modelo_fly.polyvore import asignar_slots, cargar_categorias, cargar_items

TABLA = ROOT / "data" / "mappings" / "polyvore_categorias.csv"
SALIDA = ROOT / "data" / "processed" / "hito3" / "polyvore_items.csv"
SALIDA.parent.mkdir(parents=True, exist_ok=True)

categorias = cargar_categorias(TABLA)
items = asignar_slots(cargar_items(DESTINO), categorias)
print(items.shape)

por_slot = (items.groupby("slot").agg(categorias=("categoryid", "nunique"), items=("item_key", "size"))
            .sort_values("items", ascending=False))
por_slot["% items"] = (100 * por_slot["items"] / len(items)).round(1)
display(por_slot)

excluidos = items[items["slot"] == "excluido"]
display(excluidos["motivo_exclusion"].value_counts().to_frame("items"))
print("Categorías excluidas con más ítems:")
display(excluidos["categoria"].value_counts().head(15).to_frame("items"))

(142480, 14)


,categorias,items,% items
slot,,,
excluido,230,31829,22.3
calzado,27,20521,14.4
joyeria,11,17641,12.4
bolso,13,15788,11.1
arriba,22,14884,10.4
abajo,26,14267,10.0
accesorio,31,13342,9.4
exterior,13,8520,6.0
vestido,7,5688,4.0


,items
motivo_exclusion,
"no es ropa ni accesorio (belleza, hogar, tecnologia, comida, otros)",28127
generico o ambiguo,2124
ropa interior o de dormir,704
ninos o maternidad,456
ropa de bano,418


Categorías excluidas con más ítems:


,items
categoria,
Tech Accessories,2654
Lipstick,2536
Clothing,1738
Eyeshadow,1628
Fragrance,1324
Nail Polish,1236
Mascara,956
Floral Decor,915
Home Decor,807


In [7]:
moda = items[items["slot"] != "excluido"]
por_outfit = moda.groupby("set_id")["slot"].agg(list)
resumen_outfit = pd.Series({
    "outfits originales": items["set_id"].nunique(),
    "outfits con algún ítem de moda": len(por_outfit),
    "ítems de moda por outfit (mediana)": float(por_outfit.str.len().median()),
    "ítems excluidos por outfit (media)": round(float(items.groupby("set_id")["slot"].apply(lambda s: (s == "excluido").sum()).mean()), 2),
    "ítems de categoría de hombre": int((items["genero"] == "hombre").sum()),
    "ítems con largo fijado por la categoría": int((items["largo_categoria"] != "").sum()),
}, name="valor")
display(resumen_outfit.to_frame())

items.to_csv(SALIDA, index=False, encoding="utf-8")
print("Guardado:", SALIDA.relative_to(ROOT), items.shape)

,valor
outfits originales,21889.00
outfits con algún ítem de moda,21028.00
ítems de moda por outfit (mediana),5.00
ítems excluidos por outfit (media),1.45
ítems de categoría de hombre,3441.00
ítems con largo fijado por la categoría,6819.00


Guardado: data\processed\hito3\polyvore_items.csv (142480, 14)


## 4. Paso 2 · Outfits vestibles

Se quitan del outfit los ítems excluidos en el paso 1 y se conserva el outfit solo si **se puede vestir tal cual** (regla decidida el 01/11/2026):

| Pieza | Regla |
|---|---|
| Núcleo | un **vestido** solo, **o** una prenda de **abajo** con una o dos de **arriba** (capas) |
| Calzado | **exactamente uno** |
| Exterior | como mucho uno |
| Bolso, accesorio, joyería | libres |

Polyvore permite poner alternativas en un mismo set (dos pares de zapatos, dos vestidos); esos outfits no se pueden vestir a la vez y además complican el paso 4, donde cada negativo se arma cambiando **una** prenda por otra del mismo slot. También se descartan los outfits **duplicados** (mismo conjunto de productos).

In [8]:
from modelo_fly.polyvore import SLOTS, filtrar_outfits

OUTFITS = ROOT / "data" / "processed" / "hito3" / "polyvore_outfits.csv"
outfits = filtrar_outfits(items)

embudo = pd.Series({
    "outfits originales": len(outfits),
    "con núcleo + calzado (regla amplia)": int(((outfits["vestido"] > 0) | ((outfits["arriba"] > 0) & (outfits["abajo"] > 0))).mul(outfits["calzado"] > 0).sum()),
    "vestibles (regla estricta, sin duplicados)": int(outfits["valido"].sum()),
}, name="outfits")
display(embudo.to_frame())
display(outfits["motivo_invalido"].replace("", "VÁLIDO").value_counts().to_frame("outfits"))

,outfits
outfits originales,21889
con núcleo + calzado (regla amplia),16616
"vestibles (regla estricta, sin duplicados)",15012


,outfits
motivo_invalido,
VÁLIDO,15012
"sin nucleo (falta arriba, abajo o vestido)",3402
sin calzado,878
sin items de moda,861
mas de un calzado,631
vestido junto con arriba o abajo,429
mas de un vestido,270
mas de una prenda abajo,257
mas de un exterior,102


In [9]:
validos = outfits[outfits["valido"]]
composicion = pd.Series({
    "con vestido": int((validos["vestido"] == 1).sum()),
    "arriba + abajo": int((validos["abajo"] == 1).sum()),
    "  de ellos con dos prendas arriba": int((validos["arriba"] == 2).sum()),
    "con exterior": int((validos["exterior"] == 1).sum()),
    "con bolso": int((validos["bolso"] > 0).sum()),
    "con accesorio": int((validos["accesorio"] > 0).sum()),
    "con joyería": int((validos["joyeria"] > 0).sum()),
}, name="outfits")
display(composicion.to_frame())
display(validos["items_moda"].value_counts().sort_index().rename_axis("ítems de moda").to_frame("outfits"))
display(validos["particion_oficial"].value_counts().to_frame("outfits"))

items_validos = items[items["set_id"].isin(validos["set_id"]) & (items["slot"] != "excluido")]
print("Ítems en outfits válidos:", len(items_validos), "| productos distintos:", items_validos["tid"].nunique(),
      "| sin nombre:", int((items_validos["name"] == "").sum()))
print("Productos que se repiten en más de un outfit válido:",
      int((items_validos.groupby("tid")["set_id"].nunique() > 1).sum()))

outfits.to_csv(OUTFITS, index=False, encoding="utf-8")
print("Guardado:", OUTFITS.relative_to(ROOT), outfits.shape)

,outfits
con vestido,4172
arriba + abajo,10840
de ellos con dos prendas arriba,570
con exterior,5907
con bolso,11369
con accesorio,7178
con joyería,8046


,outfits
ítems de moda,
2,62
3,844
4,3077
5,4296
6,3460
7,2016
8,1257


,outfits
particion_oficial,
train,12501
test,1693
valid,818


Ítems en outfits válidos: 81372 | productos distintos: 65366 | sin nombre: 542
Productos que se repiten en más de un outfit válido: 9362
Guardado: data\processed\hito3\polyvore_outfits.csv (21889, 15)
